In [0]:
from pyspark.sql.functions import col, input_file_name, regexp_extract, sum, count, stddev, when, row_number, min
from pyspark.sql.window import Window

RAW_CSV_PATH = "/Volumes/workspace/anima/anima_volume/raw/dynamic_ab_research/set/depression/"
FORMS_PATH = "/Volumes/workspace/anima/anima_volume/raw/dynamic_ab_research/forms.csv"

BRONZE_TABLE = "workspace.anima.bronze_eye_tracking"
SILVER_TABLE = "workspace.anima.silver_eye_tracking"
GOLD_TABLE   = "workspace.anima.gold_features"

In [0]:
from pyspark.sql.functions import col

print(f"Reading raw CSVs from {RAW_CSV_PATH}...")

bronze_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(RAW_CSV_PATH)
    .select("*", col("_metadata.file_path").alias("source_file"))
)

print("Writing to Bronze Table...")
bronze_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(BRONZE_TABLE)

display(spark.table(BRONZE_TABLE).limit(5))

In [0]:
bronze = spark.table(BRONZE_TABLE)

silver_df = bronze.withColumn("sid", regexp_extract("source_file", r"\/([^\/]+)\.csv$", 1))

forms = (
    spark.read.option("header", "true").option("inferSchema", "true").csv(FORMS_PATH)
    .select("sid", "uid", "phq-9_score", "createdAt")
)

window_spec = Window.partitionBy("uid").orderBy("createdAt")

forms_filtered = (
    forms.withColumn("session_rank", row_number().over(window_spec))
    .filter(col("session_rank") == 1)
    .drop("session_rank", "createdAt")
)

silver_joined = silver_df.join(forms_filtered, on="sid", how="inner")

print("Writing to Silver Table...")
silver_joined.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(SILVER_TABLE)

print(f"Silver Table Created. Rows: {silver_joined.count()}")

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, sum, col, when, stddev, avg, lit

silver = spark.table(SILVER_TABLE)

window_spec = Window.partitionBy("sid").orderBy("TIMESTAMP")

silver_enhanced = silver.withColumn(
    "prev_blink", 
    lag("BLINK", 1, False).over(window_spec)
).withColumn(
    "is_blink_start",
    when((col("BLINK") == True) & (col("prev_blink") == False), 1).otherwise(0)
).withColumn(
    "is_fixation_end",
    when(col("FEV") == 3, 1).otherwise(0)
)

gold_df = (
    silver_enhanced.groupBy("sid", "uid", "phq-9_score")
    .agg(
        (sum("ROW_DURATION") / 1000 / 60).alias("minutes"),
        sum("is_blink_start").alias("blink_count"),
        sum(when(col("BLINK") == True, col("ROW_DURATION")).otherwise(0)).alias("total_blink_ms"),
        sum("is_fixation_end").alias("fixation_count"),
        avg(when(col("FEV") == 3, col("FDUR"))).alias("avg_fixation_ms"),
        (stddev("FPTX") + stddev("FPTY") + stddev("FPTZ") / 3).alias("head_movement")
    )
)

gold_features = (
    gold_df
    .withColumn("blink_rate", 
        when(col("minutes") > 0, col("blink_count") / col("minutes"))
        .otherwise(0)
    )
    .withColumn("avg_blink_duration_ms", 
        when(col("blink_count") > 0, col("total_blink_ms") / col("blink_count"))
        .otherwise(0)
    )
    .withColumn("fixation_rate", 
        when(col("minutes") > 0, col("fixation_count") / (col("minutes") * 60))
        .otherwise(0)
    )
    .withColumnRenamed("head_movement", "head_movement_index")
    
    .withColumn("severity_group", 
        when(col("phq-9_score") <= 5, "1. Minimal (0-5)")
        .when((col("phq-9_score") > 5) & (col("phq-9_score") <= 10), "2. Mild (6-10)")
        .when((col("phq-9_score") > 10) & (col("phq-9_score") <= 15), "3. Moderate (11-15)")
        .otherwise("4. Severe (16+)")
    )
)

print("Writing Updated Gold Features...")
gold_features.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(GOLD_TABLE)

print("Columns in table:", gold_features.columns)
display(gold_features)

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

pdf = spark.table(GOLD_TABLE).toPandas()

pdf = pdf.sort_values('severity_group')

custom_palette = sns.color_palette("flare", n_colors=4)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.boxplot(data=pdf, x='severity_group', y='blink_rate', palette=custom_palette, ax=axes[0], showfliers=False)
axes[0].set_title("Blink Rate across Severity Levels")
axes[0].set_xlabel("Depression Severity")
axes[0].grid(axis='y', linestyle='--', alpha=0.3)

sns.boxplot(data=pdf, x='severity_group', y='avg_fixation_ms', palette=custom_palette, ax=axes[1], showfliers=False)
axes[1].set_title("Fixation Duration across Severity Levels")
axes[1].set_xlabel("Depression Severity")
axes[1].grid(axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy import stats

def visualize_severity_trend(spark_df, variable_col, title=None):
    """
    Takes a Spark DataFrame, converts to Pandas, buckets PHQ-9 into 4 groups,
    and generates a 3-panel dashboard (Violin, Swarm, ECDF).
    
    Parameters:
    - spark_df: The Gold Layer Spark DataFrame
    - variable_col: The string name of the column to analyze (e.g., 'blink_rate')
    - title: (Optional) A nice title for the charts
    """
    
    try:
        pdf = spark_df.select("phq-9_score", variable_col).toPandas()
        
        pdf = pdf.dropna(subset=[variable_col, "phq-9_score"])
        
        bins = [-1, 5, 10, 15, 100]
        labels = ["1. Minimal (0-5)", "2. Mild (6-10)", "3. Moderate (11-15)", "4. Severe (16+)"]
        pdf['severity_group'] = pd.cut(pdf['phq-9_score'], bins=bins, labels=labels)
        
        pdf = pdf.sort_values('severity_group')
        
    except Exception as e:
        print(f"Error preparing data: {e}")
        return

    if title is None:
        title = f"Analysis of {variable_col}"
        
    fig, axes = plt.subplots(1, 3, figsize=(24, 7))
    
    colors = sns.color_palette("flare", n_colors=4)

    sns.violinplot(data=pdf, x='severity_group', y=variable_col, palette=colors, 
                   inner="quart", linewidth=1.5, ax=axes[0])
    axes[0].set_title("A. Distribution Shape (Violin)", fontsize=14)
    axes[0].set_xlabel("")
    axes[0].tick_params(axis='x', rotation=15)
    
    sns.boxplot(data=pdf, x='severity_group', y=variable_col, palette=colors, 
                showfliers=False, boxprops={'alpha': 0.3}, ax=axes[1])
    sns.swarmplot(data=pdf, x='severity_group', y=variable_col, color=".2", size=2.5, alpha=0.7, ax=axes[1])
    axes[1].set_title("B. Individual Density (Swarm)", fontsize=14)
    axes[1].set_xlabel("Depression Severity")
    axes[1].tick_params(axis='x', rotation=15)

    sns.ecdfplot(data=pdf, x=variable_col, hue='severity_group', palette=colors, 
                 linewidth=3, ax=axes[2])
    axes[2].set_title("C. Cumulative Probability (ECDF)", fontsize=14)
    axes[2].grid(True, linestyle='--', alpha=0.5)

    corr, p_val = stats.spearmanr(pdf['phq-9_score'], pdf[variable_col])
    
    stats_text = f"Spearman Correlation: r={corr:.3f}, p={p_val:.4f}"
    
    plt.suptitle(f"{title}\n{stats_text}", fontsize=16)
    plt.tight_layout()
    plt.show()
    
    print(f"✅ Analysis Complete for {variable_col}")
    print(f"   {stats_text}")
    if p_val < 0.05:
        print("   ⭐ SIGNIFICANT TREND DETECTED")
    else:
        print("   ❌ No significant linear trend")

In [0]:
gold_df = spark.table(GOLD_TABLE)

visualize_severity_trend(gold_df, "blink_rate", title="Blink Rate per Minute")

visualize_severity_trend(gold_df, "avg_blink_duration_ms", title="Average Blink Duration")

visualize_severity_trend(gold_df, "fixation_rate", title="Fixation Rate (Events/Sec)")

visualize_severity_trend(gold_df, "head_movement_index", title="Head Movement Variance")

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, sum, col, when, stddev, avg, sqrt, pow, lit
from scipy import stats
import pandas as pd
import builtins 

silver = spark.table(SILVER_TABLE)
window_spec = Window.partitionBy("sid").orderBy("TIMESTAMP")

silver_enhanced = silver.withColumn(
    "prev_x", lag("BPOGX", 1).over(window_spec)
).withColumn(
    "prev_y", lag("BPOGY", 1).over(window_spec)
).withColumn(
    "step_distance", 
    sqrt(pow(col("BPOGX") - col("prev_x"), 2) + pow(col("BPOGY") - col("prev_y"), 2))
).withColumn(
    "is_blink_start",
    when((col("BLINK") == True) & (lag("BLINK", 1, False).over(window_spec) == False), 1).otherwise(0)
).withColumn(
    "is_fixation_end",
    when(col("FEV") == 3, 1).otherwise(0)
)

gold_df = (
    silver_enhanced.groupBy("sid", "uid", "phq-9_score")
    .agg(
        (sum("ROW_DURATION") / 1000 / 60).alias("minutes"),
        
        sum("is_blink_start").alias("blink_count"),
        sum(when(col("BLINK") == True, col("ROW_DURATION")).otherwise(0)).alias("total_blink_ms"),
        
        sum("is_fixation_end").alias("fixation_count"),
        avg(when(col("FEV") == 3, col("FDUR"))).alias("avg_fixation_ms"),
        
        ((stddev("FPTX") + stddev("FPTY") + stddev("FPTZ")) / 3).alias("head_movement_index"),
        
        sum("step_distance").alias("total_scanpath_pixels"),
        avg("step_distance").alias("avg_saccade_amplitude")
    )
)

gold_features = (
    gold_df
    .withColumn("blink_rate", 
        when(col("minutes") > 0, col("blink_count") / col("minutes")).otherwise(0)
    )
    .withColumn("avg_blink_duration_ms", 
        when(col("blink_count") > 0, col("total_blink_ms") / col("blink_count")).otherwise(0)
    )
    .withColumn("fixation_rate", 
        when(col("minutes") > 0, col("fixation_count") / (col("minutes") * 60)).otherwise(0)
    )
    .withColumn("scanpath_per_minute", 
        when(col("minutes") > 0, col("total_scanpath_pixels") / col("minutes")).otherwise(0)
    )
    
    .withColumn("severity_group", 
        when(col("phq-9_score") <= 5, "1. Minimal (0-5)")
        .when((col("phq-9_score") > 5) & (col("phq-9_score") <= 10), "2. Mild (6-10)")
        .when((col("phq-9_score") > 10) & (col("phq-9_score") <= 15), "3. Moderate (11-15)")
        .otherwise("4. Severe (16+)")
    )
)

gold_features.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(GOLD_TABLE)
print("✅ Gold Table Updated with Advanced Metrics.")

pdf = spark.table(GOLD_TABLE).toPandas()

metrics_to_test = [
    'blink_rate', 
    'avg_blink_duration_ms', 
    'head_movement_index', 
    'scanpath_per_minute', 
    'avg_saccade_amplitude',
    'avg_fixation_ms'
]

stats_results = []

print(f"\n{'METRIC':<25} | {'TREND (p)':<10} | {'ANOVA (p)':<10} | {'CORRELATION (r)'}")
print("-" * 75)

for metric in metrics_to_test:
    data = pdf.dropna(subset=[metric, 'phq-9_score'])
    
    if not data.empty:
        corr, p_corr = stats.spearmanr(data['phq-9_score'], data[metric])
        
        groups = [data[data['severity_group'] == g][metric] for g in sorted(data['severity_group'].unique())]
        if len(groups) > 1:
            f_stat, p_anova = stats.f_oneway(*groups)
        else:
            p_anova = 1.0
            
        star = "*" if p_corr < 0.05 else ""
        print(f"{metric:<25} | {p_corr:.4f} {star:<4} | {p_anova:.4f}     | r={corr:.3f}")
        
        stats_results.append({'metric': metric, 'p_trend': p_corr, 'r': corr})

if stats_results:
    best_metric = builtins.min(stats_results, key=lambda x: x['p_trend'])
    
    print("-" * 75)
    print(f"🏆 The strongest biomarker is: {best_metric['metric']}")
    print(f"   (Trend p-value: {best_metric['p_trend']:.5f}, Correlation: {best_metric['r']:.3f})")
    
    if best_metric['r'] < 0:
        print(f"   📉 Interpretation: As depression gets worse, {best_metric['metric']} DECREASES.")
    else:
        print(f"   📈 Interpretation: As depression gets worse, {best_metric['metric']} INCREASES.")
        
    print(f"\n👉 Recommendation: Run 'visualize_severity_trend(spark.table(GOLD_TABLE), \"{best_metric['metric']}\")' to see the chart.")

In [0]:
import matplotlib.pyplot as plt

pdf = spark.table(GOLD_TABLE).toPandas()

plt.figure(figsize=(10, 5))
plt.hist(pdf['minutes'], bins=30, color='skyblue', edgecolor='black')
plt.title("Distribution of Session Durations")
plt.xlabel("Duration (Minutes)")
plt.ylabel("Count of Users")
plt.show()

print(pdf['minutes'].describe())

In [0]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline
from pyspark.sql.functions import col

df = spark.table(GOLD_TABLE).filter(col("minutes") > 1.0)
print(f"Data count after filtering: {df.count()}")

feature_cols = [
    "blink_rate", 
    "avg_blink_duration_ms", 
    "fixation_rate", 
    "scanpath_per_minute", 
    "avg_saccade_amplitude", 
    "head_movement_index"
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw")

scaler = StandardScaler(inputCol="features_raw", outputCol="features", withStd=True, withMean=True)

lr = LinearRegression(featuresCol="features", labelCol="phq-9_score", regParam=0.1)

train_data, test_data = df.randomSplit([0.8, 0.2], seed=42)

pipeline = Pipeline(stages=[assembler, scaler, lr])
model = pipeline.fit(train_data)
predictions = model.transform(test_data)

evaluator_rmse = RegressionEvaluator(labelCol="phq-9_score", predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol="phq-9_score", predictionCol="prediction", metricName="r2")

rmse = evaluator_rmse.evaluate(predictions)
r2 = evaluator_r2.evaluate(predictions)

print("-" * 50)
print(f"MODEL RESULTS (Linear Regression)")
print(f"RMSE (Error): {rmse:.2f}")
print(f"R2 (Explained Variance): {r2:.4f}")
print("-" * 50)

lr_model = model.stages[-1]
print("Feature Importance (Weights):")
for feature, weight in zip(feature_cols, lr_model.coefficients):
    print(f"  {feature:<25}: {weight:.4f}")

if r2 < 0.1:
    print("\n⚠️ CONCLUSION: The model is struggling. Eye metrics alone might not predict the EXACT score.")
    print("   Suggestion: Try Classification (Depressed vs Healthy) next.")
else:
    print("\n✅ CONCLUSION: The model found a signal!")

In [0]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.sql.functions import col, when
from pyspark.ml import Pipeline

df = spark.table(GOLD_TABLE).filter(col("minutes") > 1.0)

data = df.withColumn("label", when(col("phq-9_score") >= 10, 1.0).otherwise(0.0))

print(f"Total valid sessions: {data.count()}")
print("Class Balance (Is the data fair?):")
data.groupBy("label").count().show()

feature_cols = [
    "blink_rate", 
    "avg_blink_duration_ms", 
    "fixation_rate", 
    "scanpath_per_minute", 
    "avg_saccade_amplitude", 
    "head_movement_index"
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw")

scaler = StandardScaler(inputCol="features_raw", outputCol="features", withStd=True, withMean=True)

lr = LogisticRegression(featuresCol="features", labelCol="label", regParam=0.1)

train_data, test_data = data.randomSplit([0.8, 0.2], seed=42)

pipeline = Pipeline(stages=[assembler, scaler, lr])
model = pipeline.fit(train_data)
predictions = model.transform(test_data)

evaluator_auc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
auc = evaluator_auc.evaluate(predictions)

evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy")
accuracy = evaluator_acc.evaluate(predictions)

evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", metricName="f1")
f1_score = evaluator_f1.evaluate(predictions)

print("-" * 50)
print("MODEL RESULTS (Classification)")
print(f"AUC Score:  {auc:.3f}  (0.5 = Random Guessing, 0.7+ = Good, 0.8+ = Excellent)")
print(f"Accuracy:   {accuracy:.1%}")
print(f"F1 Score:   {f1_score:.3f}")
print("-" * 50)

lr_model = model.stages[-1]
print("\nFeature Importance (Log Odds):")
print("(Positive (+) means higher values -> Depressed)")
print("(Negative (-) means higher values -> Healthy)")
print("-" * 40)

for feature, weight in zip(feature_cols, lr_model.coefficients):
    print(f"  {feature:<25}: {weight:.4f}")

print("\nConfusion Matrix:")
predictions.groupBy("label", "prediction").count().orderBy("label", "prediction").show()